# K ближайших соседей (KNN)

KNN — простой алгоритм, который делает прогноз по наиболее похожим объектам из обучающей выборки. Он подходит и для классификации, и для регрессии.

## Содержание
1. [Главная идея](#1-главная-идея)
2. [Расстояние и масштаб признаков](#2-расстояние-и-масштаб-признаков)
3. [Классификация](#3-классификация)
4. [Регрессия](#4-регрессия)
5. [KNN в scikit-learn](#5-knn-в-scikit-learn)
6. [Как выбрать k](#6-как-выбрать-k)
7. [Как k влияет на bias и variance](#7-как-k-влияет-на-bias-и-variance)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Главная идея

Для нового объекта KNN выполняет три действия:

1. Считает расстояния до объектов обучающей выборки.
2. Находит $k$ ближайших соседей.
3. Объединяет их ответы:
   - в классификации выбирает самый частый класс;
   - в регрессии усредняет числовые ответы.

KNN почти ничего не вычисляет во время `fit`: он в основном запоминает обучающие данные. Основная работа выполняется при `predict`. Поэтому KNN называют **ленивым алгоритмом**.

Параметр $k$ означает число соседей. Это не число классов и не число признаков.

In [ ]:
X_small = np.array([[1, 1], [2, 1], [2, 2], [5, 4], [6, 5], [5, 6]])
y_small = np.array([0, 0, 0, 1, 1, 1])
query = np.array([3.5, 3.0])

distances = np.linalg.norm(X_small - query, axis=1)
neighbor_ids = np.argsort(distances)[:3]
prediction = np.bincount(y_small[neighbor_ids]).argmax()

print('Индексы трёх соседей:', neighbor_ids)
print('Их классы:', y_small[neighbor_ids])
print('Предсказанный класс:', prediction)

plt.figure(figsize=(6, 4))
plt.scatter(X_small[:, 0], X_small[:, 1], c=y_small, cmap='coolwarm', s=80)
plt.scatter(*query, marker='X', c='gold', edgecolor='black', s=180, label='Новый объект')
for idx in neighbor_ids:
    plt.plot([query[0], X_small[idx, 0]], [query[1], X_small[idx, 1]], 'k--', alpha=0.4)
plt.legend()
plt.title('Три ближайших соседа')
plt.show()

## 2. Расстояние и масштаб признаков

Чаще всего используется евклидово расстояние:

$$d(x,z)=\sqrt{\sum_{j=1}^{p}(x_j-z_j)^2}.
$$

Если возраст измеряется десятками лет, а доход — сотнями тысяч рублей, доход будет почти полностью определять расстояние. Поэтому числовые признаки перед KNN обычно масштабируют. Самый частый вариант — `StandardScaler`.

Масштабировщик и KNN удобно объединять в один `Pipeline`. Тогда при кросс-валидации scaler обучается только на тренировочной части каждого фолда, и информация из validation не попадает в preprocessing.

Параметр `p` задаёт метрику Минковского:

- `p=2` — евклидово расстояние;
- `p=1` — манхэттенское расстояние, сумма модулей различий.

In [ ]:
point_a = np.array([20, 40_000])  # возраст, доход
point_b = np.array([40, 41_000])
print('Без масштабирования:', np.linalg.norm(point_a - point_b))

scaled_points = StandardScaler().fit_transform(np.vstack([point_a, point_b]))
print('После масштабирования:', np.linalg.norm(scaled_points[0] - scaled_points[1]))

## 3. Классификация

В обычном варианте каждый сосед имеет один голос:

$$\hat y(x)=\operatorname{mode}\{y_i: x_i\in N_k(x)\}.
$$

При `weights='distance'` близкие объекты получают больший вес, обычно пропорциональный $1/d$. Это может помочь, когда сосед на расстоянии 0.1 важнее соседа на расстоянии 3.

Для бинарной классификации часто выбирают нечётное $k$, чтобы реже получать равенство голосов. Для многоклассовой задачи ничья всё равно возможна и разрешается правилом реализации.

In [ ]:
def knn_classify_one(X_train, y_train, x, k=5):
    distances = np.linalg.norm(X_train - x, axis=1)
    neighbor_ids = np.argsort(distances)[:k]
    return np.bincount(y_train[neighbor_ids]).argmax()

X, y = make_moons(n_samples=350, noise=0.22, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
pred = np.array([knn_classify_one(X_train, y_train, x, k=7) for x in X_test])
print(f'Accuracy простой реализации: {accuracy_score(y_test, pred):.3f}')

## 4. Регрессия

В регрессии прогноз — среднее ответов соседей:

$$\hat y(x)=\frac1k\sum_{x_i\in N_k(x)}y_i.
$$

При взвешивании по расстоянию используется взвешенное среднее. KNN-регрессия хорошо описывает локальные зависимости, но не умеет продолжать тренд далеко за границы обучающих данных: прогноз остаётся комбинацией известных ответов.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X_reg = np.sort(rng.uniform(0, 8, 120)).reshape(-1, 1)
y_reg = np.sin(X_reg[:, 0]) + rng.normal(0, 0.2, len(X_reg))
grid = np.linspace(0, 8, 400).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X_reg[:, 0], y_reg, s=18, alpha=0.55, label='Данные')
for k in [1, 7, 30]:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_reg, y_reg)
    ax.plot(grid[:, 0], model.predict(grid), label=f'k={k}')
ax.legend()
ax.set_title('KNN-регрессия: увеличение k сглаживает прогноз')
plt.show()

## 5. KNN в scikit-learn

В `scikit-learn` используются два основных класса:

- `KNeighborsClassifier` — классификация;
- `KNeighborsRegressor` — регрессия.

Ключевые параметры:

| Параметр | Значение |
|---|---|
| `n_neighbors` | число соседей $k$ |
| `weights` | `uniform` или `distance` |
| `metric` | способ измерения расстояния; по умолчанию Minkowski |
| `p` | степень Minkowski: 1 — Manhattan, 2 — Euclidean |
| `algorithm` | поиск соседей: `auto`, `ball_tree`, `kd_tree`, `brute` |
| `leaf_size` | влияет на скорость и память деревьев поиска, но не меняет смысл модели |

`algorithm` относится к способу **поиска соседей**, а не к обучению дерева решений. При `auto` sklearn выбирает подходящий вариант. В большой размерности KD-tree и BallTree часто теряют преимущество, и полный перебор может оказаться конкурентоспособным.

In [ ]:
knn_pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=7, weights='distance', p=2),
)
knn_pipeline.fit(X_train, y_train)
test_pred = knn_pipeline.predict(X_test)
test_prob = knn_pipeline.predict_proba(X_test)

print(f'Test accuracy: {accuracy_score(y_test, test_pred):.3f}')
print('Форма predict_proba:', test_prob.shape)
print('Сумма вероятностей первого объекта:', test_prob[0].sum())

## 6. Как выбрать k

Маленькое $k$ создаёт сложную границу и чувствительно к отдельным объектам. Большое $k$ сильнее сглаживает решение и может игнорировать локальную структуру. Поэтому $k$ подбирают по кросс-валидации только на обучающей выборке.

Одновременно можно сравнить `weights`, метрику и другие параметры. Здесь для наглядности меняется только $k$, чтобы понимать причину изменения качества.

In [ ]:
k_values = range(1, 32, 2)
cv_accuracy = []
for k in k_values:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_accuracy.append(scores.mean())

best_k = list(k_values)[int(np.argmax(cv_accuracy))]
plt.figure(figsize=(8, 4))
plt.plot(list(k_values), cv_accuracy, marker='o')
plt.axvline(best_k, color='red', linestyle='--', label=f'лучший k={best_k}')
plt.xlabel('k')
plt.ylabel('Средняя CV accuracy')
plt.title('Подбор числа соседей')
plt.legend()
plt.show()
print('Лучшее k по CV:', best_k)

## 7. Как k влияет на bias и variance

Параметр $k$ управляет сложностью KNN:

| Значение k | Поведение модели | Bias | Variance | Типичный риск |
|---|---|---|---|---|
| Маленькое, например $k=1$ | Очень локальная и неровная граница | Низкий | Высокая | Переобучение |
| Среднее | Сохраняет сигнал и сглаживает отдельный шум | Средний | Средняя | Часто лучший баланс |
| Большое, близкое к размеру train | Почти всем объектам даётся похожий прогноз | Высокий | Низкая | Недообучение |

Почему так происходит:

- при $k=1$ прогноз зависит от одного объекта; небольшая замена обучающей выборки может сильно изменить результат — это высокая variance;
- при увеличении $k$ усреднение делает прогноз устойчивее — variance снижается;
- одновременно модель хуже замечает небольшие локальные детали — bias растёт;
- при $k=n$ классификатор почти везде предсказывает самый частый класс, а регрессор — среднее target.

Важно: train-ошибка при $k=1$ обычно минимальна, но это не означает лучшую работу на новых данных. Оптимальное $k$ выбирают по validation/CV, где проявляется баланс bias и variance.

In [ ]:
# Эмпирически оценим variance: много раз заменим обучающую выборку
# и посмотрим, насколько меняются предсказанные вероятности на одной сетке.
grid_x, grid_y = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 55),
    np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 55),
)
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]
rng = np.random.default_rng(RANDOM_STATE)
summary = []

for k in [1, 5, 25, 101]:
    probability_runs = []
    train_errors = []
    test_errors = []
    for _ in range(40):
        ids = rng.choice(len(X_train), size=len(X_train), replace=True)
        model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
        model.fit(X_train[ids], y_train[ids])
        probability_runs.append(model.predict_proba(grid_points)[:, 1])
        train_errors.append(1 - accuracy_score(y_train[ids], model.predict(X_train[ids])))
        test_errors.append(1 - accuracy_score(y_test, model.predict(X_test)))
    probability_runs = np.asarray(probability_runs)
    variance = probability_runs.var(axis=0).mean()
    summary.append((k, np.mean(train_errors), np.mean(test_errors), variance))

print(' k | train error | test error | variance прогнозов')
for k, train_error, test_error, variance in summary:
    print(f'{k:3d} | {train_error:11.3f} | {test_error:10.3f} | {variance:18.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([row[0] for row in summary], [row[1] for row in summary], 'o-', label='train error')
axes[0].plot([row[0] for row in summary], [row[2] for row in summary], 'o-', label='test error')
axes[0].set(xlabel='k', ylabel='Ошибка', title='Слишком малое и слишком большое k')
axes[0].legend()
axes[1].plot([row[0] for row in summary], [row[3] for row in summary], 'o-')
axes[1].set(xlabel='k', ylabel='Средняя variance вероятности', title='Variance уменьшается с ростом k')
plt.tight_layout()
plt.show()

На конкретной случайной выборке числа могут немного меняться, но общая закономерность сохраняется: **рост $k$ повышает bias и снижает variance**. Поэтому крайние значения обычно проигрывают умеренному $k$ на новых данных.